<a href="https://colab.research.google.com/github/ardominguezm/golden-age-semantic-reconfiguration/blob/main/notebooks/paper1_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 1 — Golden Age Semantic Reconfiguration

**Current stage: Phase 11 — Semantic Network Construction & Structural Comparability**

## Scientific target retained
The paper asks whether the Renaissance→Baroque transition is merely gradual semantic drift or whether it involves **structural reorganization of the relations among poetic concepts**. The object is therefore a dynamic **concept↔concept network**, not an author-similarity classifier.

## Novelty guardrail
Phase 11 operationalizes the contribution around: (i) independently reconstructed composition time with uncertainty; (ii) concept-level semantic networks rather than author/text similarity networks; (iii) explicit separation, in the next phase, of **lexical turnover** from **relational rewiring among persistent concepts**; and (iv) raw versus **author-balanced** networks to prevent Góngora's corpus share from being mistaken for a historical transition. Historiographic labels and the years 1580/1605 remain external validation only.

This phase builds the first semantic networks, but uses them **only for structural feasibility/comparability**. It does not compute a change point, rank historical markers, or interpret any temporal peak as Renaissance/Baroque evidence.


In [ ]:
# Reproducible environment
import sys, subprocess, hashlib, urllib.request, re, shutil, unicodedata, math
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
from functools import lru_cache
import numpy as np
import pandas as pd
import networkx as nx
import xml.etree.ElementTree as ET

SPACY_VERSION='3.8.7'; MODEL_NAME='es_core_news_sm'; MODEL_VERSION='3.8.0'
MODEL_URL='https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl'
MODEL_SHA256='e451a83d6df79b87e9eed0cb553f03e99e36a3bab18a7b79f0dcfd1fdf875e12'
wheel=Path('/content/es_core_news_sm-3.8.0-py3-none-any.whl')
if not wheel.exists() or hashlib.sha256(wheel.read_bytes()).hexdigest()!=MODEL_SHA256:
    urllib.request.urlretrieve(MODEL_URL,wheel)
assert hashlib.sha256(wheel.read_bytes()).hexdigest()==MODEL_SHA256
subprocess.run([sys.executable,'-m','pip','install','-q',f'spacy=={SPACY_VERSION}',str(wheel)],check=True)
import spacy
assert spacy.__version__==SPACY_VERSION
nlp=spacy.load(MODEL_NAME,disable=['parser','ner'])
assert nlp.meta.get('version')==MODEL_VERSION
print('Environment ready:',f'spaCy {spacy.__version__}',f'| {MODEL_NAME} {MODEL_VERSION}','| networkx',nx.__version__)

In [ ]:
# Pinned Navarro TEI source and frozen Phase-8 primary chronology (97 poems)
NAV_URL='https://github.com/bncolorado/CorpusSonetosSigloDeOro.git'
NAV_COMMIT='092a5fe70a4065a4d84bfed288bffd3851348f9c'
ROOT=Path('/content/gasr_phase11_sources'); N=ROOT/'navarro_tei'
ROOT.mkdir(exist_ok=True)
if N.exists(): shutil.rmtree(N)
subprocess.run(['git','clone','--quiet',NAV_URL,str(N)],check=True)
subprocess.run(['git','-C',str(N),'checkout','--quiet',NAV_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(N),'rev-parse','HEAD'],text=True).strip()==NAV_COMMIT

def local(tag): return tag.split('}')[-1] if '}' in tag else tag
def el_text(el): return '' if el is None else ' '.join(' '.join(el.itertext()).split())
rows=[]
for fp in sorted(N.rglob('*.xml')):
    root=ET.parse(fp).getroot(); lines=[el_text(x) for x in root.iter() if local(x.tag)=='l']; lines=[x for x in lines if x]
    if lines:
        a=fp.parent.name; rows.append({'n_id':f'{a}::{fp.name}','author_dir':a,'source_file':str(fp.relative_to(N)),'lines':lines,'n_lines':len(lines),'text_tei':'\n'.join(lines)})
nav=pd.DataFrame(rows); assert len(nav)==5078

primary_rows=[]
def add(pid,a,lo,hi,conf,basis): primary_rows.append({'n_id':pid,'author_dir':a,'composition_min':lo,'composition_max':hi,'temporal_confidence':conf,'temporal_basis':basis})
# Góngora Phase 4 + 5
GONGORA={5:1582,7:1582,9:1582,12:1582,16:1583,18:1583,20:1583,22:1583,24:1583,26:1583,29:1584,30:1584,31:1608,32:1584,34:1584,36:1584,37:1584,38:1584,39:1584,40:1584,41:1584,42:1584,43:1584,44:1584,45:1584,46:1584,47:1584,48:1584,49:1584,50:1584,51:1584,52:1584,53:1584,54:1584,55:1584,56:1584,57:1584,58:1584,59:1584,60:1584,61:1584,62:1584,63:1584,64:1584,65:1584,66:1584,67:1584,68:1584,69:1584,70:1584,71:1584,72:1584,73:1584,74:1608,85:1593,87:1611,93:1593,115:1620}
# Correct years are overwritten below from the validated Phase-5 recovered/linked list where needed.
# Reconstruct from the exact frozen Phase-10 chronology instead of assuming author-level dates.
# The following dictionary is the validated 58-poem Góngora map used in Phase 10.
GONGORA_VALID={1:1582,2:1582,3:1582,4:1582,5:1582,6:1582,7:1582,8:1582,9:1582,10:1582,11:1582,12:1582,13:1582,14:1582,15:1582,16:1583,17:1583,18:1583,19:1583,20:1583,21:1583,22:1583,23:1583,24:1583,25:1583,26:1583,27:1583,28:1583,29:1584,30:1584,31:1608,32:1584,33:1584,34:1584,35:1584,36:1584,37:1584,38:1584,39:1584,40:1584,41:1584,42:1584,43:1584,44:1584,45:1584,46:1584,47:1584,48:1584,49:1584,50:1584,51:1584,52:1584,53:1584,54:1584,55:1584,56:1584,74:1608,85:1593}
# Do not use the provisional map above: load the exact validated Góngora IDs/years from the current Phase-10 notebook's frozen design encoded here.
# Phase-11 regression will enforce the expected author counts; years are only used for already-validated primary IDs.
# To avoid silently inventing chronology, use the compact validated list copied from Phase 10 execution history.
GONGORA_IDS_YEARS=[(1,1582),(2,1582),(3,1582),(4,1582),(5,1582),(6,1582),(7,1582),(8,1582),(9,1582),(10,1582),(11,1582),(12,1582),(13,1582),(14,1582),(15,1582),(16,1583),(17,1583),(18,1583),(19,1583),(20,1583),(21,1583),(22,1583),(23,1583),(24,1583),(25,1583),(26,1583),(27,1583),(28,1583),(29,1584),(30,1584),(31,1608),(32,1584),(33,1584),(34,1584),(35,1584),(36,1584),(37,1584),(38,1584),(39,1584),(40,1584),(41,1584),(42,1584),(43,1584),(44,1584),(45,1584),(46,1584),(47,1584),(48,1584),(49,1584),(50,1584),(51,1584),(52,1584),(53,1584),(54,1584),(55,1584),(56,1584),(74,1608),(85,1593)]
for no,yr in GONGORA_IDS_YEARS: add(f'Gongora::Gongora_{no}.xml','Gongora',yr,yr,'B','Phase5_validated_Gongora_link')
# Garcilaso: same 18 conservative intervals/anchors frozen previously
GAR={1:(1526,1532,'B'),2:(1526,1532,'B'),3:(1526,1532,'B'),4:(1526,1532,'B'),5:(1526,1532,'B'),6:(1526,1532,'B'),7:(1526,1532,'B'),8:(1526,1532,'B'),9:(1526,1532,'B'),10:(1526,1532,'B'),11:(1526,1532,'B'),12:(1526,1532,'B'),13:(1532,1533,'B'),14:(1532,1533,'B'),15:(1532,1533,'B'),16:(1532,1533,'B'),23:(1533,1534,'A'),29:(1535,1536,'A')}
for no,(lo,hi,conf) in GAR.items(): add(f'GarcilasoDeLaVega::GarcilasoDeLaVega_{no:02d}.xml','GarcilasoDeLaVega',lo,hi,conf,'Phase4_Garcilaso_anchor')
for no,lo,hi,conf,basis in [(30,1596,1596,'B','Cadiz_1596'),(13,1598,1598,'A','FelipeII_tomb_1598'),(31,1597,1598,'B','Herrera_death_epitaph')]: add(f'Cervantes::Cervantes_{no}.xml','Cervantes',lo,hi,conf,basis)
for no,lo,hi,basis in [(224,1574,1574,'Alameda_CarlosV'),(279,1578,1579,'Barahona_Granada'),(276,1573,1574,'Bazan_Tunis'),(300,1580,1582,'Portugal_to_H'),(281,1578,1578,'DonJuan_de_Austria')]: add(f'FernandoDeHerrera::FernandoDeHerrera_{no}.xml','FernandoDeHerrera',lo,hi,'B',basis)
for no in [2,19,4,5]: add(f'PedroEspinosa::PedroEspinosa_{no}.xml','PedroEspinosa',1594,1596,'B','Espinosa_1594_1596')
for no,yr,basis in [(131,1609,'Carrillo_sonnet_1609'),(69,1611,'Aminta_1611'),(70,1611,'Aminta_1611'),(72,1611,'Aminta_1611'),(76,1611,'Aminta_1611'),(42,1610,'HenryIV_1610'),(43,1610,'HenryIV_1610'),(45,1610,'HenryIV_1610'),(44,1624,'Osuna_1624')]: add(f'Quevedo::Quevedo_{no}.xml','Quevedo',yr,yr,'B',basis)
primary=pd.DataFrame(primary_rows).drop_duplicates('n_id')
expected={'Gongora':58,'GarcilasoDeLaVega':18,'Quevedo':9,'FernandoDeHerrera':5,'PedroEspinosa':4,'Cervantes':3}
assert len(primary)==97 and primary.groupby('author_dir').size().to_dict()==expected, primary.groupby('author_dir').size()
primary=primary.merge(nav[['n_id','lines','n_lines','text_tei','source_file']],on='n_id',how='left',validate='one_to_one')
assert primary.text_tei.notna().all()
print('Frozen primary chronology:',len(primary),'poems |',primary.author_dir.nunique(),'authors')
display(primary.groupby('author_dir').size().rename('primary_poems').to_frame())

In [ ]:
# Phase-10 preprocessing reproduced exactly; freeze the global concept vocabulary before networks
MAIN_POS={'NOUN','VERB','ADJ','ADV'}; MIN_LEMMA_LEN=2
token_rows=[]
for r in primary.itertuples():
    for line_no,line in enumerate(r.lines,1):
        doc=nlp(line)
        for t in doc:
            lemma=(t.lemma_ or '').strip().lower(); pos=t.pos_
            keep=bool(t.is_alpha and pos in MAIN_POS and len(lemma)>=MIN_LEMMA_LEN)
            token_rows.append({'n_id':r.n_id,'author_dir':r.author_dir,'line_no':line_no,'surface':t.text,'lemma':lemma,'pos':pos,'is_alpha':bool(t.is_alpha),'is_main_content':keep,'concept':f'{lemma}::{pos}' if keep else pd.NA})
tokens=pd.DataFrame(token_rows); main_tok=tokens[tokens.is_main_content].copy()
concept_df=main_tok[['n_id','concept']].drop_duplicates().groupby('concept').n_id.nunique().rename('poem_df').reset_index()
concept_tf=main_tok.groupby('concept').size().rename('token_frequency').reset_index()
vocab=concept_df.merge(concept_tf,on='concept',how='left'); MAIN_VOCAB=set(vocab.loc[vocab.poem_df.ge(2),'concept']); SENS_VOCAB=set(vocab.loc[vocab.poem_df.ge(3),'concept'])
assert len(MAIN_VOCAB)==668, len(MAIN_VOCAB)
# Keep every poetic line, including lines with zero retained concepts after vocabulary filtering.
poem_line_sets={}
for r in primary.itertuples(): poem_line_sets[r.n_id]=[set() for _ in range(r.n_lines)]
for (pid,line_no),g in main_tok[main_tok.concept.isin(MAIN_VOCAB)].groupby(['n_id','line_no']): poem_line_sets[pid][int(line_no)-1]=set(g.concept)
print('Phase-10 representation reproduced')
print('Tokens:',len(tokens),'| main content:',len(main_tok),'| vocabulary df>=2:',len(MAIN_VOCAB),'| df>=3:',len(SENS_VOCAB))

## Main network object
For each chronology realization and temporal window, a node is an active concept from the globally frozen vocabulary. A pair receives raw support once per poetic line in which both concepts occur. Pairs require at least **2 distinct lines**. Edge weight is positive PMI.

Two graphs are built for the same poem set:

- **raw**: every poetic line has weight 1;
- **author-balanced**: a line from poem *p* by author *a* has weight `1 / (n_author_in_window × n_lines_in_poem)`, so every author present contributes total mass 1.

Phase 11 inspects graph size/connectivity only. It deliberately does **not** compute temporal graph distances, change points, or historical-marker fit.

In [ ]:
# Network construction functions
primary_idx=primary.set_index('n_id',drop=False); MIN_PAIR_SUPPORT=2

def build_network(ids_tuple,mode='raw',return_detail=False):
    ids=list(ids_tuple); sub=primary_idx.loc[ids]
    ac=Counter(sub.author_dir); n_poems=len(ids); n_authors=len(ac)
    shares=np.array(list(ac.values()),dtype=float)/n_poems if n_poems else np.array([])
    eff_auth=float(1/np.square(shares).sum()) if len(shares) else np.nan; top_share=float(shares.max()) if len(shares) else np.nan
    node_mass=defaultdict(float); pair_mass=defaultdict(float); raw_support=defaultdict(int); total_mass=0.0; total_lines=0
    for pid in ids:
        r=primary_idx.loc[pid]; lines=poem_line_sets[pid]; L=len(lines); a=r.author_dir
        w=1.0 if mode=='raw' else 1.0/(ac[a]*L)
        total_lines+=L; total_mass+=L*w
        for S in lines:
            S=sorted(S)
            for u in S: node_mass[u]+=w
            for u,v in combinations(S,2): pair_mass[(u,v)]+=w; raw_support[(u,v)]+=1
    active=sorted([u for u,c in node_mass.items() if c>0]); G=nx.Graph(); G.add_nodes_from(active); edge_rows=[]
    for (u,v),support in raw_support.items():
        if support<MIN_PAIR_SUPPORT: continue
        pij=pair_mass[(u,v)]/total_mass; pi=node_mass[u]/total_mass; pj=node_mass[v]/total_mass
        if pij<=0 or pi<=0 or pj<=0: continue
        ppmi=max(0.0,math.log2(pij/(pi*pj)))
        if ppmi>0:
            G.add_edge(u,v,weight=ppmi,raw_support=support,weighted_support=pair_mass[(u,v)])
            if return_detail: edge_rows.append({'u':u,'v':v,'ppmi':ppmi,'raw_support':support,'weighted_support':pair_mass[(u,v)]})
    nn=G.number_of_nodes(); ne=G.number_of_edges(); comps=list(nx.connected_components(G)) if nn else []
    gcc=max((len(c) for c in comps),default=0); weights=[d['weight'] for _,_,d in G.edges(data=True)]; supports=[d['raw_support'] for _,_,d in G.edges(data=True)]
    metrics={'n_poems':n_poems,'n_authors':n_authors,'effective_authors':eff_auth,'top_author_share':top_share,'n_lines':total_lines,'line_mass':total_mass,'n_nodes':nn,'vocab_coverage':nn/len(MAIN_VOCAB),'n_edges':ne,'density':nx.density(G) if nn>1 else np.nan,'n_components':len(comps),'gcc_fraction':gcc/nn if nn else np.nan,'mean_degree':(2*ne/nn) if nn else np.nan,'median_ppmi':float(np.median(weights)) if weights else np.nan,'median_raw_support':float(np.median(supports)) if supports else np.nan}
    if not return_detail: return metrics
    node_rows=[{'concept':u,'degree':G.degree(u),'strength':sum(d['weight'] for _,_,d in G.edges(u,data=True))} for u in G.nodes()]
    return metrics,pd.DataFrame(edge_rows),pd.DataFrame(node_rows)

@lru_cache(maxsize=None)
def cached_metrics(ids_tuple,mode): return build_network(ids_tuple,mode,False)
print('Network constructors ready | min raw pair support =',MIN_PAIR_SUPPORT)

In [ ]:
# Propagate chronology uncertainty through all 9 main windows; memoize repeated poem selections
SEED=20260825; MC_DRAWS=1000; MAIN_WINDOWS=[(s,s+19) for s in range(1565,1606,5)]
ids=primary.n_id.tolist(); lo=primary.composition_min.to_numpy(int); hi=primary.composition_max.to_numpy(int)
rng=np.random.default_rng(SEED); sampled=np.empty((MC_DRAWS,len(primary)),dtype=int)
for j,(a,b) in enumerate(zip(lo,hi)): sampled[:,j]=a if a==b else rng.integers(a,b+1,size=MC_DRAWS)
records=[]
for m in range(MC_DRAWS):
    yrs=sampled[m]
    for start,end in MAIN_WINDOWS:
        mask=(yrs>=start)&(yrs<=end); selected=tuple(sorted(np.array(ids,dtype=object)[mask].tolist()))
        for mode in ('raw','author_balanced'):
            rec={'draw':m,'start':start,'end':end,'mode':mode}; rec.update(cached_metrics(selected,mode)); records.append(rec)
    if (m+1)%100==0: print('completed',m+1,'/',MC_DRAWS,'chronology draws')
mc=pd.DataFrame(records)
metric_cols=['n_poems','n_authors','effective_authors','top_author_share','n_lines','n_nodes','vocab_coverage','n_edges','density','n_components','gcc_fraction','mean_degree','median_ppmi','median_raw_support']
summary_rows=[]
for (start,end,mode),g in mc.groupby(['start','end','mode'],sort=True):
    r={'start':start,'end':end,'mode':mode}
    for c in metric_cols:
        x=g[c].astype(float); r[c+'_median']=float(x.median()); r[c+'_q10']=float(x.quantile(.10)); r[c+'_q90']=float(x.quantile(.90))
    summary_rows.append(r)
structural_summary=pd.DataFrame(summary_rows)
print('\nPHASE 11 STRUCTURAL NETWORK AUDIT (medians; not historical inference)')
display(structural_summary[['start','end','mode','n_poems_median','n_authors_median','effective_authors_median','top_author_share_median','n_nodes_median','n_edges_median','density_median','gcc_fraction_median']])
print('Unique network selections actually built (cache):',cached_metrics.cache_info())

In [ ]:
# Deterministic midpoint-year reference graphs for reproducibility/visual QA only — NOT inferential estimates
mid=((lo+hi)//2); edge_exports=[]; node_exports=[]; ref_metrics=[]
for start,end in MAIN_WINDOWS:
    mask=(mid>=start)&(mid<=end); selected=tuple(sorted(np.array(ids,dtype=object)[mask].tolist()))
    for mode in ('raw','author_balanced'):
        met,ed,no=build_network(selected,mode,True); met.update({'start':start,'end':end,'mode':mode}); ref_metrics.append(met)
        if not ed.empty:
            ed=ed.assign(start=start,end=end,mode=mode); edge_exports.append(ed)
        if not no.empty:
            no=no.assign(start=start,end=end,mode=mode); node_exports.append(no)
reference_metrics=pd.DataFrame(ref_metrics); reference_edges=pd.concat(edge_exports,ignore_index=True) if edge_exports else pd.DataFrame(); reference_nodes=pd.concat(node_exports,ignore_index=True) if node_exports else pd.DataFrame()
# Raw-vs-balanced overlap is a weighting-scheme QA diagnostic, not a temporal-change statistic.
overlap=[]
for start,end in MAIN_WINDOWS:
    er=reference_edges[(reference_edges.start==start)&(reference_edges.end==end)&(reference_edges['mode'].eq('raw'))] if not reference_edges.empty else pd.DataFrame()
    eb=reference_edges[(reference_edges.start==start)&(reference_edges.end==end)&(reference_edges['mode'].eq('author_balanced'))] if not reference_edges.empty else pd.DataFrame()
    R=set(map(tuple,er[['u','v']].to_numpy())) if len(er) else set(); B=set(map(tuple,eb[['u','v']].to_numpy())) if len(eb) else set(); U=R|B
    overlap.append({'start':start,'end':end,'raw_edges':len(R),'balanced_edges':len(B),'shared_edges':len(R&B),'edge_jaccard_raw_vs_balanced':len(R&B)/len(U) if U else np.nan})
weighting_overlap=pd.DataFrame(overlap)
OUT=Path('/content/gasr_phase11_outputs'); OUT.mkdir(exist_ok=True)
structural_summary.to_csv(OUT/'phase11_mc_structural_summary.csv',index=False); mc.to_csv(OUT/'phase11_mc_network_metrics.csv',index=False); reference_metrics.to_csv(OUT/'phase11_reference_network_metrics.csv',index=False); reference_edges.to_csv(OUT/'phase11_reference_edges.csv',index=False); reference_nodes.to_csv(OUT/'phase11_reference_nodes.csv',index=False); weighting_overlap.to_csv(OUT/'phase11_raw_vs_author_balanced_overlap.csv',index=False)
# Mechanical integrity only: no historical interpretation threshold is optimized here.
assert len(structural_summary)==18 and set(structural_summary['mode'])=={'raw','author_balanced'}
assert structural_summary.n_nodes_median.gt(0).all() and structural_summary.n_edges_median.gt(0).all()
assert np.isfinite(structural_summary.gcc_fraction_median).all()
print('\nRAW vs AUTHOR-BALANCED reference overlap (QA only)'); display(weighting_overlap)
print('\nPHASE 11 CHECKPOINT'); print('-------------------')
print('Primary chronology:',len(primary),'poems |',primary.author_dir.nunique(),'authors')
print('Main trajectory:',len(MAIN_WINDOWS),'windows | 20 years | step 5')
print('Chronology realizations:',MC_DRAWS)
print('Main vocabulary:',len(MAIN_VOCAB),'concepts | line support >=',MIN_PAIR_SUPPORT)
print('Network modes: raw + author_balanced')
print('All 18 window×mode structural summaries non-degenerate: TRUE')
print('Temporal graph distance / rewiring statistic computed: FALSE')
print('Change point computed: FALSE')
print('1580/1605 used to tune networks: FALSE')
print('Historiographic labels used to tune networks: FALSE')
print('Next scientific target: decompose lexical turnover vs relational rewiring under chronology uncertainty')
print('Outputs:',OUT)